In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


In [25]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [26]:
import keras
print(keras.__version__)

3.10.0


<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
1. Data Loading
</font>
</h2>

In [ ]:
train = pd.read_csv('train.csv',engine='pyarrow')
test = pd.read_csv('test.csv', engine='pyarrow') 

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
2. Data Cleaning
</font>
</h2>


In [ ]:
train = train.drop(columns='user')
test = test.drop(columns='user')


In [ ]:
missing_checkout = train[train['checkOut_date'].isna()]
missing_checkin = train[train['checkIn_date'].isna()]

In [ ]:
ratio_zero_checkin = (
    missing_checkin['is_booking'].eq(0).mean()
)

ratio_zero_checkout = (
    missing_checkout['is_booking'].eq(0).mean()
)
THRESHOLD = 0.99
if ratio_zero_checkin >= THRESHOLD:
    train = train[train['checkIn_date'].notna()]
    print('Rows with missing checkIn_date were dropped')
else:
    print('Use another strategy for checkIn_date')

if ratio_zero_checkout >= THRESHOLD:
    train = train[train['checkOut_date'].notna()]
    print('Rows with missing checkOut_date were dropped')
else:
    print('Use another strategy for checkOut_date')


Rows with missing checkIn_date were dropped
Rows with missing checkOut_date were dropped


In [31]:
train.isna().mean()['destination_distance']

0.35762694533684297

In [32]:
mean = (
    train
    .groupby(['user_location_city', 'destination'])['destination_distance']
    .mean()
)
mapped_values = train.set_index(
    ['user_location_city', 'destination']
).index.map(mean)

train['destination_distance'] = train['destination_distance'].fillna(
    pd.Series(mapped_values, index=train.index)
)

mapped_values_test = test.set_index(
    ['user_location_city', 'destination']
).index.map(mean)

test['destination_distance'] = test['destination_distance'].fillna(
    pd.Series(mapped_values_test, index=test.index)
)


In [33]:
train[train['destination_distance'] == 0]

,user_location_country,user_location_region,user_location_city,destination_distance,search_date,is_mobile,is_package,channel,search_count,checkIn_date,...,n_adults,n_children,n_rooms,destination,destination_type,hotel_continent,hotel_country,hotel_market,hotel_category,is_booking


In [ ]:
train['destination_distance'].fillna(0, inplace=True) 
test['destination_distance'].fillna(0, inplace=True) 

In [35]:
train.isna().sum()

user_location_country    0
user_location_region     0
user_location_city       0
destination_distance     0
search_date              0
is_mobile                0
is_package               0
channel                  0
search_count             0
checkIn_date             0
checkOut_date            0
n_adults                 0
n_children               0
n_rooms                  0
destination              0
destination_type         0
hotel_continent          0
hotel_country            0
hotel_market             0
hotel_category           0
is_booking               0
dtype: int64

<h3 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
3. Feature Engineering</font>
</h3>


In [ ]:
time_columns = ['search_date', 'checkIn_date', 'checkOut_date']
for col in time_columns:
    train[col] = pd.to_datetime(train[col])
for col in time_columns:
    test[col] = pd.to_datetime(test[col])

In [ ]:
test[time_columns].dtypes


search_date       datetime64[s]
checkIn_date     datetime64[ns]
checkOut_date    datetime64[ns]
dtype: object

In [ ]:

duration = train['checkOut_date'] - train['checkIn_date'] # TODO
train['duration'] = duration.dt.days # TODO
duration = test['checkOut_date'] - test['checkIn_date'] # TODO
test['duration'] = duration.dt.days# TODO


days_between = train['checkIn_date']-train['search_date'] # TODO
train['days_between'] = days_between.dt.days # TODO
days_between = test['checkIn_date']-test['search_date'] # TODO
test['days_between'] = days_between.dt.days # TODO


In [39]:
train['search_date_hour'] = train['search_date'].dt.hour
train['search_date_dayofweek'] = train['search_date'].dt.dayofweek
train['checkIn_date_dayofweek'] = train['checkIn_date'].dt.dayofweek
train['search_date_month'] = train['search_date'].dt.month
train['checkIn_date_month'] = train['checkIn_date'].dt.month

test['search_date_hour'] = test['search_date'].dt.hour
test['search_date_dayofweek'] = test['search_date'].dt.dayofweek
test['checkIn_date_dayofweek'] = test['checkIn_date'].dt.dayofweek
test['search_date_month'] = test['search_date'].dt.month
test['checkIn_date_month'] = test['checkIn_date'].dt.month

In [40]:
is_booked = train[train['is_booking'] == 1] #TODO
not_booked = train[train['is_booking'] == 0]

<h3 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
4. Exploratory Data Analysis (EDA)
</font>
</h3>


In [ ]:
not_counts = not_booked['search_date_hour'].value_counts(normalize=True).sort_index()
is_counts = is_booked['search_date_hour'].value_counts(normalize=True).sort_index()
trace_not_booked = go.Bar(
    x=list(range(24)),
    y=not_counts.values,
    name='Not Booked'
)

trace_is_booked = go.Bar(
    x=list(range(24)),
    y=is_counts.values,
    name='Booked'
)

data = [trace_is_booked, trace_not_booked]

layout = go.Layout(
    xaxis=dict(title='Search Hour', tickangle=45, automargin=True),
    yaxis=dict(title='Frequency')
)

fig = go.Figure(data=data, layout=layout)
fig.show()
fig.write_json('./search_hour.json')

In [ ]:
not_counts = (
    not_booked['checkIn_date_dayofweek']
    .value_counts(normalize=True)
    .sort_index()
    
)
is_counts = (
    is_booked['checkIn_date_dayofweek']
    .value_counts(normalize=True)
    .sort_index()

)


trace_not_booked = go.Bar(x=[0,1,2,3,4,5,6],y=not_counts.values , name='Not Booked') # TODO
trace_is_booked = go.Bar(x=[0,1,2,3,4,5,6],y=is_counts.values , name='Booked') # TODO

ticktext = ['دوشنبه','سه‌شنبه','چهارشنبه','پنج‌شنبه','جمعه','شنبه','یکشنبه'] # TODO

data = [trace_is_booked, trace_not_booked]

layout = go.Layout(
    xaxis=dict(title='Day of Week', tickangle=45, automargin=True,
               tickvals = [0,1,2,3,4,5,6], ticktext= ticktext
 ),
    yaxis=dict(title='Frequency'),
)

fig = go.Figure(data=data, layout=layout)
fig.show()
fig.write_json('./checkIn_day.json')

In [ ]:
not_counts = (
    not_booked['checkIn_date_month']
    .value_counts(normalize=True)
    .sort_index()
    
)

is_counts = (
    is_booked['checkIn_date_month']
    .value_counts(normalize=True)
    .sort_index()

)

trace_not_booked = go.Bar(
    x=np.arange(0, 12),
    y=not_counts.values,
    name='Not Booked'
)

trace_is_booked = go.Bar(
    x=np.arange(0, 12),
    y=is_counts.values,
    name='Booked'
)

data = [trace_is_booked, trace_not_booked]

ticktext = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

layout = go.Layout(
    xaxis=dict(title='Month', tickangle=45, automargin=True,
             ticktext = ticktext ,tickvals = np.arange(0,12)),
    yaxis=dict(title='Frequency')
)

fig = go.Figure(data=data, layout=layout)
fig.show()
fig.write_json('./checkIn_date_month.json')

In [ ]:
not_counts = (
    not_booked['days_between']
    .value_counts(normalize=True)
    .sort_index()
)

is_counts = (
    is_booked['days_between']
    .value_counts(normalize=True)
    .sort_index()
)
trace_not_booked = go.Scatter(
    y=not_counts,
    name='Not Booked',
    opacity=0.5,
    mode='lines'
)

trace_is_booked = go.Scatter(
    y=is_counts,
    name='Booked',
    mode='lines'
)
data = [trace_is_booked, trace_not_booked]

layout = go.Layout(
    xaxis=dict(
        title='Days between search and checking time',
        tickangle=45,
        automargin=True,
        range=[0, 700]
    ),
    yaxis=dict(title='Frequency')
)

fig = go.Figure(data=data, layout=layout)
fig.show()
fig.write_json('./days_between.json')


In [ ]:


not_counts = (
    not_booked['duration']
    .value_counts(normalize=True)
    .sort_index()
)

is_counts = (
    is_booked['duration']
    .value_counts(normalize=True)
    .sort_index()
)

trace_not_booked = go.Scatter(
    x=not_counts.index,
    y=not_counts.values,
    name='Not Booked',
    opacity=0.5,
    mode='lines'
)

trace_is_booked = go.Scatter(
    x=is_counts.index,
    y=is_counts.values,
    name='Booked',
    mode='lines'
)

data = [trace_is_booked, trace_not_booked]

layout = go.Layout(
    xaxis=dict(
        title='Length of Stay',
        tickangle=45,
        automargin=True,
        range=[0, 170]
    ),
    yaxis=dict(title='Frequency')
)

fig = go.Figure(data=data, layout=layout)
fig.show()
fig.write_json('./los.json')


<h3 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
5. Preprocessing
</font>
</h3>

In [ ]:
train['is_abroad'] = np.where(train['user_location_country'] != train['hotel_country'], 1 ,0) # TODO
test['is_abroad'] = np.where(test['user_location_country'] != test['hotel_country'], 1 ,0) # TODO

In [ ]:
train.drop(['hotel_country','user_location_country','user_location_region','user_location_city'] , axis=1 , inplace=True)
test.drop(['hotel_country','user_location_country','user_location_region','user_location_city'] , axis=1 , inplace=True)
train.drop(time_columns, axis=1 , inplace=True)
test.drop(time_columns , axis=1 , inplace=True)
test = test.iloc[:, 1:]

In [ ]:
from sklearn.model_selection import train_test_split
x= train.drop('is_booking',axis=1)
y=train['is_booking']
X_train, X_valid, y_train, y_valid = train_test_split(
    x, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  
)
print("Train set class ratio:", y_train.value_counts(normalize=True))
print("Valid set class ratio:", y_valid.value_counts(normalize=True))

from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1]),
    y=y_train
)

class_weight = {
    0: class_weights[0],
    1: class_weights[1]
}


Train set class ratio: is_booking
0    0.913012
1    0.086988
Name: proportion, dtype: float64
Valid set class ratio: is_booking
0    0.913012
1    0.086988
Name: proportion, dtype: float64


<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
5. Model Development
</font>
</h2>

In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2

model = Sequential()

model.add(Dense(
    128,
    input_shape=(X_train.shape[1],)
))
model.add(BatchNormalization())
model.add(keras.layers.Activation('relu'))
model.add(Dropout(0.3))


model.add(Dense(
    64,
))
model.add(BatchNormalization())
model.add(keras.layers.Activation('relu'))
model.add(Dropout(0.25))
model.add(Dense(
    32
))
model.add(BatchNormalization())
model.add(keras.layers.Activation('relu'))
model.add(Dropout(0.25))

model.add(Dense(1, activation='sigmoid'))


d:\conda_envs\quera\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[AUC(name='auc')]
)

<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
6. Training
</font>
</h2>

In [53]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=10,
    batch_size=128,
    class_weight=class_weight,
    callbacks=[checkpoint_cb]
)


Epoch 1/10
47555/47555 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.6372 - loss: 0.6563
Epoch 1: val_loss improved from inf to 0.87730, saving model to best_model.keras
47555/47555 ━━━━━━━━━━━━━━━━━━━━ 170s 3ms/step - auc: 0.6372 - loss: 0.6563 - val_auc: 0.6723 - val_loss: 0.8773
Epoch 2/10
47543/47555 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7103 - loss: 0.6046
Epoch 2: val_loss improved from 0.87730 to 0.85341, saving model to best_model.keras
47555/47555 ━━━━━━━━━━━━━━━━━━━━ 155s 3ms/step - auc: 0.7103 - loss: 0.6046 - val_auc: 0.6898 - val_loss: 0.8534
Epoch 3/10
47548/47555 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7189 - loss: 0.5962
Epoch 3: val_loss improved from 0.85341 to 0.54433, saving model to best_model.keras
47555/47555 ━━━━━━━━━━━━━━━━━━━━ 162s 3ms/step - auc: 0.7189 - loss: 0.5962 - val_auc: 0.7320 - val_loss: 0.5443
Epoch 4/10
47538/47555 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7237 - loss: 0.5915
Epoch 4: val_loss improved from 0.54433 to 0.45840, saving model to be

<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
7. Evaluation
</font>
</h2>



In [54]:
from sklearn.metrics import roc_auc_score

y_pred_proba = model.predict(X_valid).ravel()
auc_score = roc_auc_score(y_valid, y_pred_proba)

print("Validation ROC-AUC:", auc_score)


47555/47555 ━━━━━━━━━━━━━━━━━━━━ 43s 907us/step
Validation ROC-AUC: 0.7417151186880045


<h3 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
8. Prediction & Submission</font>
</h3>



In [55]:
test_pred_proba = model.predict(test).ravel()

10182/10182 ━━━━━━━━━━━━━━━━━━━━ 9s 930us/step


In [57]:
submission = pd.DataFrame({
    'prediction': test_pred_proba
})
submission

,prediction
0,0.562122
1,0.621820
2,0.581646
3,0.682389
4,0.561403
...,...
325816,0.723944
325817,0.711471
325818,0.720710
325819,0.673474
